# Malay-English Lemmatizer Token Demo

Activate the five comparison systems, enter one sentence, and inspect each system's token-level output.

In [10]:
from pathlib import Path
import gc
import os
import re
import sys

import pandas as pd

# Resolve the repository root when this notebook is opened from the src/ folder.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "src" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from lemmatizer_systems import build_system

SYSTEM_KEYS = ["malaya_naive", "sastrawi", "stem_lstm_512", "sailor2", "llama"]
TOKEN_RE = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*|\d+|[^\w\s]", re.UNICODE)

print(f"Project root: {PROJECT_ROOT}")
print(f"Systems: {', '.join(SYSTEM_KEYS)}")

Project root: C:\Users\User\Desktop\NLP
Systems: malaya_naive, sastrawi, stem_lstm_512, sailor2, llama


## Activate all systems

This cell constructs every requested system. Large model checkpoints are loaded only when their prediction is run.

In [11]:
systems = {}
activation_errors = {}

for key in SYSTEM_KEYS:
    try:
        systems[key] = build_system(key)
        print(f"Activated: {key}")
    except Exception as exc:
        activation_errors[key] = f"{type(exc).__name__}: {exc}"
        print(f"Could not activate {key}: {activation_errors[key]}")

print(f"Ready: {len(systems)}/{len(SYSTEM_KEYS)} systems")

Activated: malaya_naive
Activated: sastrawi
[stem_lstm_512] loading mesolitica/stem-lstm-512 via malaya.stem.huggingface ...
Activated: stem_lstm_512
Activated: sailor2
Activated: llama
Ready: 5/5 systems


## Enter a sentence

In [12]:
# {"sentence": "Kalau tak ada daun kesum, kemungkinan besar asam laksa akan jadi tak sedap.", "target": [{"surface": "Kalau", "lemma": "kalau"}, {"surface": "tak", "lemma": "tidak"}, {"surface": "ada", "lemma": "ada"}, {"surface": "daun", "lemma": "daun"}, {"surface": "kesum", "lemma": "sum"}, {"surface": ",", "lemma": ","}, {"surface": "kemungkinan", "lemma": "mungkin"}, {"surface": "besar", "lemma": "besar"}, {"surface": "asam", "lemma": "asam"}, {"surface": "laksa", "lemma": "laksa"}, {"surface": "akan", "lemma": "akan"}, {"surface": "jadi", "lemma": "jadi"}, {"surface": "tak", "lemma": "tidak"}, {"surface": "sedap", "lemma": "sedap"}, {"surface": ".", "lemma": "."}]}

sentence = input("Enter a Malay-English sentence: ").strip()
if not sentence:
    raise ValueError("Please enter a non-empty sentence.")

print(f"Input: {sentence}")

Input: Kalau tak ada daun kesum, kemungkinan besar asam laksa akan jadi tak sedap.


## Token-level outputs

Each row is one token. `surface` is the input token and `lemma` is that system's output.

In [13]:
def release_system_resources(system):
    if hasattr(system, "_stemmer"):
        system._stemmer = None
    if hasattr(system, "_model"):
        system._model = None
    if hasattr(system, "_tokenizer"):
        system._tokenizer = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


# Baseline systems require target surfaces for identical tokenisation.
# Lemmas are set to the surface because this is an inference demo, not gold evaluation.
surfaces = TOKEN_RE.findall(sentence)
rows = [{
    "sentence": sentence,
    "target": [{"surface": token, "lemma": token} for token in surfaces],
}]

token_rows = []
run_errors = dict(activation_errors)

for key, system in systems.items():
    try:
        predictions = system.predict_batch(rows)
        prediction = predictions[0] if predictions else None
        if prediction is None:
            token_rows.append({
                "System": system.name,
                "Token": "<unparseable>",
                "Surface": "",
                "Lemma": "",
                "Status": "unparseable output",
            })
        else:
            token_number = 0
            for item in prediction:
                if not isinstance(item, dict) or "surface" not in item or "lemma" not in item:
                    continue
                token_number += 1
                surface = str(item["surface"])
                lemma = str(item["lemma"])
                token_rows.append({
                    "System": system.name,
                    "Token": token_number,
                    "Surface": surface,
                    "Lemma": lemma,
                    "Status": "changed" if surface.lower() != lemma.lower() else "unchanged",
                })
    except Exception as exc:
        run_errors[key] = f"{type(exc).__name__}: {exc}"
        token_rows.append({
            "System": key,
            "Token": "<error>",
            "Surface": "",
            "Lemma": "",
            "Status": run_errors[key],
        })
    finally:
        release_system_resources(system)

output_table = pd.DataFrame(token_rows)
display(output_table)

if run_errors:
    print("Systems with activation or inference errors:")
    display(pd.DataFrame([{"System": key, "Reason": reason} for key, reason in run_errors.items()]))

c:\Users\User\Desktop\NLP\.venv-1\Lib\site-packages\malaya\model\stem.py:28: FutureWarning: Possible nested set at position 3
  or re.findall(_expressions['ic'], word.lower())


[sailor2_ft] loading ./trained_models/sailor2_malay_lemmatizer ...


Loading weights: 100%|██████████| 387/387 [00:00<00:00, 14068.26it/s]


[sailor2_ft] loaded in 2.8s on cuda:0
[llama_ft] loading ./trained_models/llama_malay_lemmatizer ...


Loading weights: 100%|██████████| 255/255 [00:00<00:00, 19612.13it/s]


[llama_ft] loaded in 3.5s on cuda:0


,System,Token,Surface,Lemma,Status
0,malaya_naive,1,Kalau,kalau,unchanged
1,malaya_naive,2,tak,tak,unchanged
2,malaya_naive,3,ada,ada,unchanged
3,malaya_naive,4,daun,daun,unchanged
4,malaya_naive,5,kesum,sum,changed
...,...,...,...,...,...
70,llama_ft,11,akan,akan,unchanged
71,llama_ft,12,jadi,jadi,unchanged
72,llama_ft,13,tak,tidak,changed
73,llama_ft,14,sedap,sedap,unchanged


In [19]:
rows_by_system = {}
for row in token_rows:
    rows_by_system.setdefault(row["System"], []).append(row)

sentence_rows = [{
    "Type": "INPUT",
    "Model": "Original sentence",
    "Sentence": sentence,
}]

for system_name, model_rows in rows_by_system.items():
    valid_tokens = [row for row in model_rows if not str(row["Token"]).startswith("<")]
    if not valid_tokens:
        output_sentence = f"[{model_rows[0]['Status']}]"
    else:
        output_sentence = " ".join(row["Lemma"] for row in valid_tokens)
    sentence_rows.append({
        "Type": "OUTPUT",
        "Model": system_name,
        "Sentence": output_sentence,
    })

sentence_table = pd.DataFrame(sentence_rows)
pd.set_option("display.max_colwidth", None)
display(
    sentence_table.style
    .set_properties(
        subset=["Type", "Model", "Sentence"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "word-wrap": "break-word",
            "max-width": "800px",
        },
    )
)

,Type,Model,Sentence
0,INPUT,Original sentence,"Kalau tak ada daun kesum, kemungkinan besar asam laksa akan jadi tak sedap."
1,OUTPUT,malaya_naive,"kalau tak ada daun sum , mungkin besar asam laksa akan jadi tak dap ."
2,OUTPUT,sastrawi,"kalau tak ada daun sum , mungkin besar asam laksa akan jadi tak sedap ."
3,OUTPUT,stem_lstm_512,"kalau tak ada daun sum , kemungkinan besar asam laksa akan jadi tak sedap ."
4,OUTPUT,sailor2_ft,"kalau tak ada daun sum , mungkin besar asam laksa akan jadi tak sedap ."
5,OUTPUT,llama_ft,"kalau tidak ada daun sum , mungkin besar asam laksa akan jadi tidak sedap ."
